# Imports

In [4]:
import os, re, tempfile
from pathlib import Path
import subprocess as sp
import pandas as pd
# import argparse
# import gzip
# import os
# import shutil

# import sys
# import tempfile

from typing import List, Tuple, Optional
# 

from schrodinger import structure as sstruct

SCHRO = "/opt/schrodinger2025-2"

In [5]:
outdir = Path("dldocking")
outdir.mkdir(exist_ok=True)

## Data

In [6]:
extra_holos_featuresd = pd.read_pickle("../training_data/7.Extra_set/features.pkl")
extra_holos_featuresd

{'7gqu':     Residues                                                          \
          pdb label_entity_id label_asym_id label_seq_id auth_asym_id   
 0       7gqu               1             A           12            A   
 1       7gqu               1             A           13            A   
 2       7gqu               1             A           14            A   
 3       7gqu               1             A           15            A   
 4       7gqu               1             A           16            A   
 ..       ...             ...           ...          ...          ...   
 414     7gqu               1             A          426            A   
 415     7gqu               1             A          427            A   
 416     7gqu               1             A          428            A   
 417     7gqu               1             A          429            A   
 418     7gqu               1             A          430            A   
 
                                   Label 

In [7]:
extra_holos_sites = {k: v for k, v in pd.read_pickle("../training_data/7.Extra_set/news_sites.pkl").items() if k in extra_holos_featuresd}
extra_holos_sites

{'7gqu': [{'mod':      label_comp_id label_asym_id label_entity_id label_seq_id  \
   3420           X1L             D               4            .   
   
        pdbx_PDB_ins_code auth_seq_id auth_comp_id auth_asym_id  \
   3420                 ?        1002          X1L            A   
   
        pdbx_PDB_model_num pdbx_label_index pdbx_sifts_xref_db_name  \
   3420                  1             1002                       ?   
   
        pdbx_sifts_xref_db_acc pdbx_sifts_xref_db_num pdbx_sifts_xref_db_res  
   3420                      ?                      ?                      ?  ,
   'site':    label_comp_id label_asym_id label_entity_id label_seq_id pdbx_PDB_ins_code  \
   0            VAL             A               1           55                 ?   
   1            MET             A               1           56                 ?   
   2            ALA             A               1           57                 ?   
   3            THR             A               1         

In [8]:
apos = pd.read_pickle(f"pdb_ensemble/apos.pkl")
overlaps = pd.read_pickle(f"pdb_ensemble/predictions.pkl")
origapos = pd.read_pickle("../models/other_tools_apos/models_lenient_labelling.pkl")["model5"]

origholos = pd.read_pickle("../models/other_tools/models.pkl")["model5"]

#### 8jp0 - 8sgi

In [9]:
overlaps["8jp0"].pop("8sgi")

{'pocket': 'pocket2',
 'site_in_pocket': 0.53125,
 'pocket_in_site': 0.3695652173913043,
 'site':    label_comp_id label_asym_id label_entity_id label_seq_id pdbx_PDB_ins_code  \
 0            GLU             A               1          132                 ?   
 1            THR             A               1          133                 ?   
 2            VAL             A               1          134                 ?   
 3            SER             A               1          135                 ?   
 4            LEU             A               1          137                 ?   
 5            THR             A               1          138                 ?   
 6            ALA             A               1          141                 ?   
 7            HIS             A               1          200                 ?   
 8            VAL             A               1          203                 ?   
 9            VAL             A               1          206                 ?   
 

In [10]:
seqsf = f"cofolding/seqs.pkl"

seqs = pd.read_pickle(seqsf)

In [11]:
seqsdf = pd.DataFrame({k: {k1: v1 for k1, v1 in v.items() if k1 != "site"} for k, v in seqs.items()}).T
seqsdf

,seq,seqmin,seqmax,site_seqids,ccd,smiles
7gqu,FLWPAPNEEQVTCLKMYFGHSSFKPVQWKVIHSVLEERRDNVAVMA...,12,430,"(44, 45, 46, 177, 178, 179, 180, 181, 185, 199...",X1L,CC(c1ncc(c(n1)Oc2ccccc2)C(=O)NC(CCS(=O)(=O)C)C...
7yg5,NIVRKYAKKLIDWPPFEYMILATIIANCIVLALEQHLPEDDKTPMS...,76,1871,"(134, 135, 138, 139, 142, 278, 279, 282, 283, ...",TOR,CC1(OC2COC3(C(C2O1)OC(O3)(C)C)COS(=O)(=O)N)C
8aq6,MVFTLEDFVGDWRQTAGYNLDQVLEQGGVSSLFQNLGVSVTPIQRI...,11,181,"(11, 12, 13, 43, 45, 57, 59, 61, 83, 84, 85, 9...",NT0,c1ccc(cc1)Cc2c(ncc(n2)c3ccccc3)NC(=O)Cc4ccco4
8f4s,SSQAWQPGVAMPNLYKMQRMLLEKCDLQNYGDSATLPKGIMMNVAK...,4,301,"(68, 70, 71, 96, 98, 111, 113, 114, 115, 116, ...",XDU,c1cc(c(cc1Cl)Cl)C=Cc2cc(nc(n2)O)C(F)(F)F
8jp0,GSYYCKKGVILPIWEPQDPSFGDKIARATVYFVAMVYMFLGVSIIA...,51,937,"(82, 83, 84, 85, 87, 88, 91, 150, 153, 156, 15...",EKY,CCOc1ccc(c(c1)N)Oc2ccc(cc2)OCc3cc(ccc3F)F
8qni,QAAADRRTVEKTWKLMDKVVRLCQNPKLQLKNSPPYILDILPDTYQ...,5,387,"(104, 107, 108, 109, 111, 112, 115, 181, 182, ...",W7R,CC1CN(CCO1)Cc2ccccc2C(=O)NC3C(=O)Nc4cnccc4C(=N...
8uk6,PERSMFSEGFLGDLHKPGEEPQMYPELLEEHKKFICDKVYTRFPPE...,233,812,"(49, 50, 51, 52, 55, 280, 281, 282, 283, 284, ...",WVK,c1cc(sc1)C=CC(=O)Nc2ncccn2
8v81,RSPLEKASVVSKLFFSWTRPILRKGYRQRLELSDIYQIPSVDSADN...,1,1438,"(90, 93, 308, 309, 342, 343, 345, 346, 351, 91...",WG5,c1cc(cc(c1)N2C(=O)C(=Cc3ccc(cc3)C(=O)O)SC2=S)C...
9dnm,TRVFKKASPNGKLTVYLGKRDFVDHIDLVDPVDGVVLVDPEYLKER...,6,492,"(129, 344, 345, 346, 347, 348, 379, 380, 381, ...",ODN,CC1C2CCC3C45COC(C3(C2O)C1=O)(C(C4C(CCC5O)(C)C)O)O


In [12]:
bioemuensdir = Path("bioemu_8uk6/msmclust/dihedrals")

In [13]:
bioemuensresults = pd.read_pickle(bioemuensdir / "predictions.pkl")
bioemuensresults

{'frame_0': {'pocket': 'pocket1',
  'overlap': 0.9047619047619048,
  'merge':    label_comp_id label_asym_id pdbx_PDB_ins_code auth_asym_id  \
  0            GLY             A                 ?            A   
  1            PHE             A                 ?            A   
  2            LEU             A                 ?            A   
  3            HIS             A                 ?            A   
  4            HIS             A                 ?            A   
  5            GLY             A                 ?            A   
  6            THR             A                 ?            A   
  7            ILE             A                 ?            A   
  8            MET             A                 ?            A   
  9            LYS             A                 ?            A   
  10           ILE             A                 ?            A   
  11           TRP             A                 ?            A   
  12           ARG             A                 ?   

# Consensus residues

In [14]:
guide_resf = outdir / "guide_res.pkl"

if guide_resf.exists():
    guide_res = pd.read_pickle(guide_resf)
else:
    guide_res = {}

## Calculation

## Results

In [15]:
guide_res_df = pd.DataFrame(guide_res).T
guide_res_df

,seqids,all_seqids,mean_overlap,top_3_overlap,top_3_overlaps,overlaps,topres
7gqu,"(47, 204, 323)","[47, 204, 323]",0.264212,1,label_comp_id label_asym_id label_entity_id ...,"[0.14864864864864866, 0.06896551724137931, 0.4...",pdbx_sifts_xref_db_acc pdbx_sifts_xref_db_nu...
7yg5,"(710, 1710, 1742)","[399, 401, 403, 404, 405, 407, 408, 409, 451, ...",0.0,0,"Empty DataFrame Columns: [label_comp_id, label...","[0.0, 0.0, 0.0]",pdbx_sifts_xref_db_acc pdbx_sifts_xref_db_...
8aq6,"(60, 79, 94)","[40, 42, 60, 79, 94]",0.114211,0,"Empty DataFrame Columns: [label_comp_id, label...","[0.07142857142857142, 0.0, 0.3333333333333333,...",pdbx_sifts_xref_db_acc pdbx_sifts_xref_db_nu...
8f4s,"(130, 132, 149)","[70, 71, 98, 130, 132, 133, 149]",0.246201,1,label_comp_id label_asym_id label_entity_id ...,"[0.5263157894736842, 0.4230769230769231, 0.0, ...",pdbx_sifts_xref_db_acc pdbx_sifts_xref_db_nu...
8qni,"(183, 199, 203)","[183, 199, 203]",0.427736,1,label_comp_id label_asym_id label_entity_id ...,"[0.5517241379310345, 0.6086956521739131, 0.043...",pdbx_sifts_xref_db_acc pdbx_sifts_xref_db_nu...
8v81,"(261, 264, 265)","[261, 264, 265]",0.116819,0,"Empty DataFrame Columns: [label_comp_id, label...","[0.13978494623655913, 0.0, 0.0, 0.0, 0.0, 0.0,...",pdbx_sifts_xref_db_acc pdbx_sifts_xref_db_nu...
9dnm,"(121, 122, 386)","[121, 122, 164, 386, 394]",0.247056,0,"Empty DataFrame Columns: [label_comp_id, label...","[0.01694915254237288, 0.0, 0.4, 0.6, 0.0, 0.88...",pdbx_sifts_xref_db_acc pdbx_sifts_xref_db_nu...
8jp0,"(54, 82, 826)","[50, 54, 82, 83, 84, 87, 88, 90, 91, 148, 150,...",0.422506,1,label_comp_id label_asym_id label_entity_id ...,"[0.18888888888888888, 0.5625, 0.5161290322580645]",pdbx_sifts_xref_db_acc pdbx_sifts_xref_db_n...
8uk6,"(52, 58, 283)","[52, 55, 58, 246, 273, 283]",0.480159,2,label_comp_id label_asym_id label_entity_id ...,"[0.9047619047619048, 0.6190476190476191, 0.523...",label_seq_id 0 52 1 283 4...


# Functions

## General

In [16]:
def run(cmd: List[str], cwd: Optional[str] = None, **kwargs) -> None:
    print(f"[CMD] {cmd if 'shell' in kwargs else ' '.join(cmd)}")
    try:
        p = sp.run(cmd, check=True, cwd=cwd, text=True, capture_output=True, **kwargs)
        print(p.stdout, p.stderr)
    except sp.CalledProcessError as e:
        print("[CMD FAILED]", e.returncode, e.cmd)
        print("[STDOUT]\n", e.stdout)
        print("[STDERR]\n", e.stderr)
        raise
    return


def which(tool: str) -> str:
    """Return full path to a Schrödinger CLI tool."""
    p = Path(SCHRO) / tool
    if p.exists():
        return str(p)
    # Some tools live at the root (e.g., $SCHRODINGER/prepwizard), others under subdirs.
    for sub in ["", "utilities", "shape_screen", "glide", "sitemap", "ligprep", "epik", "vsw"]:
        pp = Path(SCHRO) / sub / tool
        if pp.exists():
            return str(pp)
    return tool  # fallback on PATH


def ensure_outdir(d: Path):
    d.mkdir(parents=True, exist_ok=True)
    print(f"[INFO] Output directory: {d}")

In [17]:
run(
    " ".join((
        SCHRO + "/run xglide.py",
        "-doc"
    )), 
    shell=True
)

[CMD] /opt/schrodinger2025-2/run xglide.py -doc

########################################################################
#                                                                      #
#                  XGlide input file documentation                     #
#                                                                      #
#    The input file is based on keyword/value pairs, though certain    #
#    keywords can accept multiple values).  The notes below            #
#    the keywords, with explanations of their purpose and accepted     #
#    values.                                                           #
#                                                                      #
########################################################################

####################
# Input structures #
####################

COMPLEX <file/dir>[,<ligand_asl>,<constraint-strings>]
#    Can be a Maestro or PDB file, or a directory.  If a directory, all of the
#    Maestro and PDB f

## ProtPrep

In [18]:
cmd = [
    which("prepwizard"),
    "--help"
]
run(cmd)

[CMD] /opt/schrodinger2025-2/utilities/prepwizard --help
usage: $SCHRODINGER/utilities/prepwizard [options] inputfile outputfile
    Input file should be in Maestro or PDB format.
    Output file should be in Maestro or PDB format.
    

Required:
  in_file               Input structure file (Maestro or PDB)
  out_file              Output structure file (Maestro or PDB)

Preprocess:
  -nopreprocess         Skip pre-processing (useful if only ProtAssign or
                        Impref is desired)
  -reference_st_file REFERENCE_ST_FILE
                        File containing reference structure to align to.
  -reference_pdbid REFERENCE_PDBID
                        PDB ID of the structure to align to.
  -nobondorders         Don't assign bond orders to het groups
  -noccd                Don't use the Chemical Component Dictionary when
                        assigning bond orders.
  -assign_all_residues  Assign bond orders to all residues (by default
                        residues ar

In [19]:
def prepare_protein(infile: Path, out, cwd: Path) -> Path:
    """Protein Preparation Wizard (prepwizard) → MAE"""
    prep_out =  f"{out}_prep.mae" #outdir /
    if (cwd / prep_out).exists():
        return
    prepwizard = which("prepwizard")
    
    cmd = [
        prepwizard,
        "-WAIT -HOST localhost:8",
        "-assign_all_residues", #  Assign bond orders to all residues (by default residues are only processed if they only have single order bonds). If CCD is used, existing bond orders and formal charges will be overriden for residues that are present in the CCD.
        "-rehtreat", # Delete and re-add hydrogens (will reset PDB atom names)
        "-disulfides", # Create bonds to proximal Sulfurs (delete hydrogens as needed)
        "-mse", # Convert Selenomethionine residues to Methionines
        "-max_states 1", # Maximum number of het states to generate for each protein complex.
        "-captermini", # Add terminal oxygens to polypeptides, Cap termini
        # "-minimize_adj_h", # Energy minimize all adjustable hydrogens (titratable hydrogens, water hydrogens, hydroxyls, thiols, and ASN/GLN carboxamide hydrogens).
        "-f S-OPLS",
        "-epik_pH 7.4 -epik_pHt 2.0 -samplewater -include_epik_states -propka_pH 7.4  -rmsd 0.3 -watdist 5.0",
        
        # "-noprotassign",   # speed-up; set as needed
        # "-noimpref",       # we’ll still do a restrained minimization
        
        str(infile.absolute()),
        str(prep_out),
    ]
    run(" ".join(cmd), cwd=cwd, shell=True)
    return #prep_out

# Blind

In [17]:
glidedir = Path("glide")
glidedir.mkdir(exist_ok=True)

In [19]:
dockings = {}

for holo, holod in overlaps.items():
    # if holo in ("7yg5", "8v81"):
    #     continue
        
    path = glidedir / holo
    path.mkdir(exist_ok = True)

    smif = path / f"{holo}.smi"
    smif.write_text(seqs[holo]["smiles"])
    
    dockings[holo] = {
        "files": [],
        "smiles": smif
    }

    if not (path / f"{holo}_prep.mae").exists():
        (path / holo).mkdir(exist_ok=True)
        dockings[holo]["files"].append(f"../training_data/7.Extra_set/cifs/{holo}.cif")
    
    refapo = next(apo for apo in apos[holo] if apo in os.listdir("../training_data/8.Apos/Extra_set/features"))
    if not (path / f"{refapo}_prep.mae").exists():
        (path / refapo).mkdir(exist_ok=True)
        dockings[holo]["files"].append(f"../training_data/8.Apos/Extra_set/cifs/{refapo}.cif")
    
    for apo in holod.keys():
        (path / apo).mkdir(exist_ok=True)
        if not (path / f"{refapo}_prep.mae").exists():
            dockings[holo]["files"].append(f"pdb_ensemble/predictions/{apo}/{apo}.cif")



holo = "8uk6"
path = glidedir / holo
path.mkdir(exist_ok = True)

smif = path / f"{holo}.smi"
smif.write_text(seqs[holo]["smiles"])

dockings[holo] = {
        "files": [],
        "smiles": smif
    }

if not (path / f"{holo}_prep.mae").exists():
    (path / holo).mkdir(exist_ok=True)
    dockings[holo]["files"].append(f"../training_data/7.Extra_set/cifs/{holo}.cif")

for apo in bioemuensresults:
    if not (path / f"{apo}_prep.mae").exists():
        (path / apo).mkdir(exist_ok=True)
        dockings[holo]["files"].append(str(bioemuensdir / f"predictions/{apo}/{apo}.cif"))

 

dockings

{'7gqu': {'files': ['../training_data/7.Extra_set/cifs/7gqu.cif',
   '../training_data/8.Apos/Extra_set/cifs/6yhr.cif',
   'pdb_ensemble/predictions/7gqt/7gqt.cif',
   'pdb_ensemble/predictions/8yle/8yle.cif',
   'pdb_ensemble/predictions/8pfp/8pfp.cif',
   'pdb_ensemble/predictions/7gqs/7gqs.cif'],
  'smiles': PosixPath('glide/7gqu/7gqu.smi')},
 '7yg5': {'files': ['../training_data/7.Extra_set/cifs/7yg5.cif',
   '../training_data/8.Apos/Extra_set/cifs/7xlq.cif',
   'pdb_ensemble/predictions/8epm/8epm.cif',
   'pdb_ensemble/predictions/8epl/8epl.cif'],
  'smiles': PosixPath('glide/7yg5/7yg5.smi')},
 '8aq6': {'files': ['../training_data/7.Extra_set/cifs/8aq6.cif',
   '../training_data/8.Apos/Extra_set/cifs/5b0u.cif',
   'pdb_ensemble/predictions/7snr/7snr.cif',
   'pdb_ensemble/predictions/7sny/7sny.cif',
   'pdb_ensemble/predictions/7snw/7snw.cif',
   'pdb_ensemble/predictions/7sns/7sns.cif',
   'pdb_ensemble/predictions/7vsx/7vsx.cif',
   'pdb_ensemble/predictions/8aqh/8aqh.cif',
   '

In [35]:
for holo, holod in dockings.items():
    path = glidedir / holo

    if (path / f"xglide/{holo}_topcomplexes.maegz").exists():
        continue

    preps = path / "preps"
    preps.mkdir(exist_ok=True)
    xout = path / "xglide"
    xout.mkdir(exist_ok=True)
    
    for file in holod["files"]:
        f = Path(file).resolve() # this resolves the symlink of 8uk6 frames and makes their name *_updated
        prepare_protein(f, out=f.stem, cwd=preps)

    xglide_inp = xout / f"{holo}.inp"    
    xglide_inp.write_text(f"""
RECEPTOR	{preps.absolute()}
ALIGN	FALSE
LIGAND	{holod['smiles'].absolute()}
GRIDGEN_GRID_CENTER	SELF
PPREP FALSE
SITEMAP	TRUE
SITEMAP_MAXSITES    5
GRIDGEN_INNERBOX	10
GRIDGEN_OUTERBOX	26.0
LIGPREP	TRUE
LIGPREP_EPIK	TRUE
DOCK_PRECISION	SP
DOCK_WRITE_XP_DESC	FALSE
DOCK_POSE_OUTTYPE	poseviewer
NATIVEONLY	FALSE
DOCK_LIG_VSCALE	0.80
SKIP_DOCKING	FALSE
GOOD_RMSD	2.0
GENERATE_TOP_COMPLEXES	5
    """)

    run(
        " ".join((
            SCHRO + "/run xglide.py",
            xglide_inp.name,
            "-WAIT -HOST localhost:32 -TMPLAUNCHDIR"
        )),
        cwd=xout, 
        shell=True
    )

    counts = {}
    with sstruct.StructureReader(xout / f"{holo}_topcomplexes.maegz") as r:
        for i, st in enumerate(r, 1):
            grid = st.property.get("s_i_glide_gridfile", "") # 'xglide__[6yhr|frame_3]_prep_site1__grid'
            try:
                apo, site = re.search(rf"{holo}__([^_]+(?:_\d+)?)_prep_(site\d+)", grid).groups()
            except AttributeError:
                if holo == "8uk6": # frame_* of 8uk6 have _updated added to their basename
                    apo, site = re.search(rf"{holo}__([^_]+(?:_\d+)?)_updated_prep_(site\d+)", grid).groups()
                else:
                    apo = grid.split("_")[2]
                    grid = "grid"
            except:
                raise
            rank = counts.get(apo, 0) + 1
            if rank == 6: continue
            fname = path / apo / f"{apo}_rank{rank}_pose{i}_{site}.pdb"
            with sstruct.StructureWriter(str(fname)) as w: w.append(st)
            counts[apo] = rank

# Guided

## Grid

In [25]:
def build_grid(
    grid_inp: Path,
    grid_zip: Path,
    receptor_mae: Path,
    center: Tuple[float, float, float],
):
    inner = 10
    outer = 26
    x, y, z = center
    contents = f"""FORCEFIELD   OPLS_2005
GRID_CENTER   {x:.8f}, {y:.8f}, {z:.8f}
GRIDFILE   {grid_zip.absolute()}
INNERBOX   {inner}, {inner}, {inner}
OUTERBOX   {outer}, {outer}, {outer}
RECEP_FILE   {receptor_mae.absolute()}
"""
    grid_inp.write_text(contents)
    glide = which("glide")
    run([glide, "-WAIT", str(grid_inp.absolute())], cwd=grid_zip.parent)
    return

## LigPrep

## Dock

In [18]:
dockings = pd.read_pickle(glidedir / "glide-guided_dockings.pkl")
dockings

{'7gqu': {'7gqu': {'ciff': '../training_data/7.Extra_set/cifs/7gqu.cif',
   'pfile': '../training_data/7.Extra_set/pockets/7gqu/7gqu_out/7gqu_out.cif',
   'pocket': '1',
   'pocket_cog': [6.964241027832031, 26.42584991455078, 16.23856544494629],
   'guide_cog': [11.84740161895752, 24.034400939941406, 21.308034896850586]},
  '6yhr': {'ciff': '../training_data/8.Apos/Extra_set/cifs/6yhr.cif',
   'pfile': '../training_data/8.Apos/Extra_set/pockets/6yhr/6yhr_out/6yhr_out.cif',
   'pocket': '1',
   'pocket_cog': [-17.198780059814453,
    -19.426149368286133,
    32.649749755859375],
   'guide_cog': [-9.501700401306152, -14.058333396911621, 39.8582649230957]},
  '7gqt': {'ciff': 'pdb_ensemble/predictions/7gqt/7gqt.cif',
   'pfile': 'pdb_ensemble/predictions/7gqt/7gqt/7gqt_out/7gqt_out.cif',
   'pocket': '5',
   'pocket_cog': [11.266744613647461, -2.0004360675811768, -5.426126956939697],
   'guide_cog': [6.413934230804443, -3.6677331924438477, -8.020732879638672]},
  '8yle': {'ciff': 'pdb_ens

In [26]:
for holo, holod in dockings.items():
    for guide in ("pocket", "guide"):
        path = glidedir / f"{holo}_{guide}"
        path.mkdir(exist_ok=True)
    
        if (path / f"xglide/{holo}_topcomplexes.maegz").exists():
            continue
        
        grids = path / "grids"
        grids.mkdir(exist_ok=True)
        xout = path / "xglide"
        xout.mkdir(exist_ok=True)

        gridsl = ""
        for apo, apod in holod.items():
            apopath = path / apo
            apopath.mkdir(exist_ok=True)

            grid_zip = grids / f"{apo}_grid.zip"
            if not grid_zip.exists():
                build_grid(
                    grid_inp = grids / f"{apo}_grid.inp",
                    grid_zip = grid_zip,
                    receptor_mae = glidedir / holo / "preps" / (f"{apo}_prep.mae" if (holo != "8uk6" or holo == apo) else f"{apo}_updated_prep.mae"),
                    center = apod[f"{guide}_cog"],
                )
            gridsl += f"GRID	{grid_zip.absolute()}\n"

        xglide_inp = xout / f"{holo}.inp"    
        xglide_inp.write_text(f"""
{gridsl}
ALIGN	FALSE
LIGAND	{(glidedir / holo / f"{holo}.smi").absolute()}
GRIDGEN_GRID_CENTER	AUTO
GRIDGEN_INNERBOX	10
GRIDGEN_OUTERBOX	26.0
PPREP FALSE
LIGPREP	TRUE
LIGPREP_EPIK	TRUE
DOCK_PRECISION	SP
DOCK_WRITE_XP_DESC	FALSE
DOCK_POSE_OUTTYPE	poseviewer
DOCK_POSES_PER_LIG   5
NATIVEONLY	FALSE
DOCK_LIG_VSCALE	0.80
SKIP_DOCKING	FALSE
GOOD_RMSD	2.0
GENERATE_TOP_COMPLEXES	5
        """)
    
        run(
            " ".join((
                SCHRO + "/run xglide.py",
                xglide_inp.name,
                "-WAIT -HOST localhost:16 -TMPLAUNCHDIR"
            )),
            cwd=xout, 
            shell=True
        )

        counts = {}
        with sstruct.StructureReader(xout / f"{holo}_topcomplexes.maegz") as r:
            for i, st in enumerate(r, 1):
                apo = (
                    st.property.get("s_i_glide_gridfile", "grid")
                    .rsplit("_", 1)[0]
                    .replace("_updated", "").replace("updated", "")
                )# '8pfp_grid' or frame_0_grid
                rank = counts.get(apo, 0) + 1
                if rank == 6: continue
                fname = path / apo / f"{apo}_rank{rank}.pdb"
                with sstruct.StructureWriter(str(fname)) as w: w.append(st)
                counts[apo] = rank

[CMD] /opt/schrodinger2025-2/glide -WAIT /home/fnerin/Desktop/AlloPockets/ensembles/glide/8uk6_pocket/grids/frame_0_grid.inp
JobId: avogadro-0-68dcef28
ExitStatus: finished
 
[CMD] /opt/schrodinger2025-2/glide -WAIT /home/fnerin/Desktop/AlloPockets/ensembles/glide/8uk6_pocket/grids/frame_1_grid.inp
JobId: avogadro-0-68dcef48
ExitStatus: finished
 
[CMD] /opt/schrodinger2025-2/glide -WAIT /home/fnerin/Desktop/AlloPockets/ensembles/glide/8uk6_pocket/grids/frame_2_grid.inp
JobId: avogadro-0-68dcef67
ExitStatus: finished
 
[CMD] /opt/schrodinger2025-2/glide -WAIT /home/fnerin/Desktop/AlloPockets/ensembles/glide/8uk6_pocket/grids/frame_3_grid.inp
JobId: avogadro-0-68dcef86
ExitStatus: finished
 
[CMD] /opt/schrodinger2025-2/glide -WAIT /home/fnerin/Desktop/AlloPockets/ensembles/glide/8uk6_pocket/grids/frame_4_grid.inp
JobId: avogadro-0-68dcefa5
ExitStatus: finished
 
[CMD] /opt/schrodinger2025-2/glide -WAIT /home/fnerin/Desktop/AlloPockets/ensembles/glide/8uk6_pocket/grids/frame_5_grid.inp


# Guided - bigger grid

In [ ]:
glidedir = Path("glide_bigger")

## Grid

In [25]:
def build_grid(
    grid_inp: Path,
    grid_zip: Path,
    receptor_mae: Path,
    center: Tuple[float, float, float],
):
    inner = 20
    outer = 30
    x, y, z = center
    contents = f"""FORCEFIELD   OPLS_2005
GRID_CENTER   {x:.8f}, {y:.8f}, {z:.8f}
GRIDFILE   {grid_zip.absolute()}
INNERBOX   {inner}, {inner}, {inner}
OUTERBOX   {outer}, {outer}, {outer}
RECEP_FILE   {receptor_mae.absolute()}
"""
    grid_inp.write_text(contents)
    glide = which("glide")
    run([glide, "-WAIT", str(grid_inp.absolute())], cwd=grid_zip.parent)
    return

## LigPrep

## Dock

In [18]:
dockings = pd.read_pickle(glidedir / "glide-guided_dockings.pkl")
dockings

{'7gqu': {'7gqu': {'ciff': '../training_data/7.Extra_set/cifs/7gqu.cif',
   'pfile': '../training_data/7.Extra_set/pockets/7gqu/7gqu_out/7gqu_out.cif',
   'pocket': '1',
   'pocket_cog': [6.964241027832031, 26.42584991455078, 16.23856544494629],
   'guide_cog': [11.84740161895752, 24.034400939941406, 21.308034896850586]},
  '6yhr': {'ciff': '../training_data/8.Apos/Extra_set/cifs/6yhr.cif',
   'pfile': '../training_data/8.Apos/Extra_set/pockets/6yhr/6yhr_out/6yhr_out.cif',
   'pocket': '1',
   'pocket_cog': [-17.198780059814453,
    -19.426149368286133,
    32.649749755859375],
   'guide_cog': [-9.501700401306152, -14.058333396911621, 39.8582649230957]},
  '7gqt': {'ciff': 'pdb_ensemble/predictions/7gqt/7gqt.cif',
   'pfile': 'pdb_ensemble/predictions/7gqt/7gqt/7gqt_out/7gqt_out.cif',
   'pocket': '5',
   'pocket_cog': [11.266744613647461, -2.0004360675811768, -5.426126956939697],
   'guide_cog': [6.413934230804443, -3.6677331924438477, -8.020732879638672]},
  '8yle': {'ciff': 'pdb_ens

In [26]:
for holo, holod in dockings.items():
    for guide in ("pocket", "guide"):
        path = glidedir / f"{holo}_{guide}"
        path.mkdir(exist_ok=True)
    
        if (path / f"xglide/{holo}_topcomplexes.maegz").exists():
            continue
        
        grids = path / "grids"
        grids.mkdir(exist_ok=True)
        xout = path / "xglide"
        xout.mkdir(exist_ok=True)

        gridsl = ""
        for apo, apod in holod.items():
            apopath = path / apo
            apopath.mkdir(exist_ok=True)

            grid_zip = grids / f"{apo}_grid.zip"
            if not grid_zip.exists():
                build_grid(
                    grid_inp = grids / f"{apo}_grid.inp",
                    grid_zip = grid_zip,
                    receptor_mae = glidedir / holo / "preps" / (f"{apo}_prep.mae" if (holo != "8uk6" or holo == apo) else f"{apo}_updated_prep.mae"),
                    center = apod[f"{guide}_cog"],
                )
            gridsl += f"GRID	{grid_zip.absolute()}\n"

        xglide_inp = xout / f"{holo}.inp"    
        xglide_inp.write_text(f"""
{gridsl}
ALIGN	FALSE
LIGAND	{(glidedir / holo / f"{holo}.smi").absolute()}
GRIDGEN_GRID_CENTER	AUTO
GRIDGEN_INNERBOX	25
GRIDGEN_OUTERBOX	40
PPREP FALSE
LIGPREP	TRUE
LIGPREP_EPIK	TRUE
DOCK_PRECISION	SP
DOCK_WRITE_XP_DESC	FALSE
DOCK_POSE_OUTTYPE	poseviewer
DOCK_POSES_PER_LIG   5
NATIVEONLY	FALSE
DOCK_LIG_VSCALE	0.80
SKIP_DOCKING	FALSE
GOOD_RMSD	2.0
GENERATE_TOP_COMPLEXES	5
        """)
    
        run(
            " ".join((
                SCHRO + "/run xglide.py",
                xglide_inp.name,
                "-WAIT -HOST localhost:16 -TMPLAUNCHDIR"
            )),
            cwd=xout, 
            shell=True
        )

        counts = {}
        with sstruct.StructureReader(xout / f"{holo}_topcomplexes.maegz") as r:
            for i, st in enumerate(r, 1):
                apo = (
                    st.property.get("s_i_glide_gridfile", "grid")
                    .rsplit("_", 1)[0]
                    .replace("_updated", "").replace("updated", "")
                )# '8pfp_grid' or frame_0_grid
                rank = counts.get(apo, 0) + 1
                if rank == 6: continue
                fname = path / apo / f"{apo}_rank{rank}.pdb"
                with sstruct.StructureWriter(str(fname)) as w: w.append(st)
                counts[apo] = rank

[CMD] /opt/schrodinger2025-2/glide -WAIT /home/fnerin/Desktop/AlloPockets/ensembles/glide/8uk6_pocket/grids/frame_0_grid.inp
JobId: avogadro-0-68dcef28
ExitStatus: finished
 
[CMD] /opt/schrodinger2025-2/glide -WAIT /home/fnerin/Desktop/AlloPockets/ensembles/glide/8uk6_pocket/grids/frame_1_grid.inp
JobId: avogadro-0-68dcef48
ExitStatus: finished
 
[CMD] /opt/schrodinger2025-2/glide -WAIT /home/fnerin/Desktop/AlloPockets/ensembles/glide/8uk6_pocket/grids/frame_2_grid.inp
JobId: avogadro-0-68dcef67
ExitStatus: finished
 
[CMD] /opt/schrodinger2025-2/glide -WAIT /home/fnerin/Desktop/AlloPockets/ensembles/glide/8uk6_pocket/grids/frame_3_grid.inp
JobId: avogadro-0-68dcef86
ExitStatus: finished
 
[CMD] /opt/schrodinger2025-2/glide -WAIT /home/fnerin/Desktop/AlloPockets/ensembles/glide/8uk6_pocket/grids/frame_4_grid.inp
JobId: avogadro-0-68dcefa5
ExitStatus: finished
 
[CMD] /opt/schrodinger2025-2/glide -WAIT /home/fnerin/Desktop/AlloPockets/ensembles/glide/8uk6_pocket/grids/frame_5_grid.inp


# Guided - IFD

In [20]:
glidedir = Path("glide_ifd")
glidedir.mkdir(exist_ok=True)
(glidedir / "ligpreps").mkdir(exist_ok=True)

## LigPrep

In [21]:
cmd = [
    which("ligprep"),
    "-help"
]
run(cmd)

[CMD] /opt/schrodinger2025-2/ligprep -help
usage: ligprep [options] (-ismi|-icsv|-imae|-isd) infile (-osd|-omae|-osmi|-ocsv) outfile

General:
  -h, -help             Print brief help message.
  -long_help            Print exhaustive help message.
  -inp <filename>       Read arguments from the supplied input file.
  -sif_docs             List supported simplified input file format (SIF)
                        keywords.

Ionization:
  -epik                 Use Epik Classic for ionization and tautomerization
                        (Recommended, overrides -i).
  -epikx                Use Epik for ionization and tautomerization
                        (Recommended, overrides -i).
  -emb, -epik_metal_binding
                        Run Epik with the metal_binding option so that states
                        appropriate for interactions with metal ions in
                        protein binding pockets are also generated.
  -i {0,1,2}            Ionization treatment: 0 - do not neutraliz

In [44]:
def prepare_ligands(smiles: Path, holo: str, cwd: Path): # ligprep_inp: Path, out_mae: Path,
    # -> Path: #  pH: float = 7.4, dpH: float = 1.0
    """
    LigPrep with Epik states; outputs a single MAEGZ of prepared ligands.
    Accepts SDF/MAE/SMILES (SMILES must be in a .smi or via -ismi).
    """
    outmae = (cwd / f'{holo}.mae').absolute()
    ligprep_inp = cwd / f'{holo}.inp'
    ligprep_inp.write_text(f"""
INPUT_FILE_NAME   {smiles.absolute()}
MAX_ATOMS   500
FORCE_FIELD   16
EPIK   yes
EPIKX   no
EPIK_METAL_BINDING   no
INCLUDE_ORIGINAL_STATE   no
DETERMINE_CHIRALITIES   no
IGNORE_CHIRALITIES   yes
NUM_STEREOISOMERS   32
OUT_MAE   {outmae}
    """)

    # Detect SMILES input by extension
    # is_smiles = ligand_in.suffix.lower() in [".smi", ".smiles", ".can"] # has to be .smi
    ligprep = which("ligprep")
    cmd = [ligprep, "-WAIT", "-inp", str(ligprep_inp.absolute())]
    # Epik-like states around target pH
    # cmd += ["-ph", f"{pH}", "-pht", f"{dpH}", "-s", "1"]  # 1 conformer per state (adjust as needed)
    run(cmd, cwd=cwd)
    return outmae

## Dock

In [45]:
dockings = pd.read_pickle(glidedir.parent / "glide" / "glide-guided_dockings.pkl")
dockings

{'7gqu': {'7gqu': {'ciff': '../training_data/7.Extra_set/cifs/7gqu.cif',
   'pfile': '../training_data/7.Extra_set/pockets/7gqu/7gqu_out/7gqu_out.cif',
   'pocket': '1',
   'pocket_cog': [6.964241027832031, 26.42584991455078, 16.23856544494629],
   'guide_cog': [11.84740161895752, 24.034400939941406, 21.308034896850586]},
  '6yhr': {'ciff': '../training_data/8.Apos/Extra_set/cifs/6yhr.cif',
   'pfile': '../training_data/8.Apos/Extra_set/pockets/6yhr/6yhr_out/6yhr_out.cif',
   'pocket': '1',
   'pocket_cog': [-17.198780059814453,
    -19.426149368286133,
    32.649749755859375],
   'guide_cog': [-9.501700401306152, -14.058333396911621, 39.8582649230957]},
  '7gqt': {'ciff': 'pdb_ensemble/predictions/7gqt/7gqt.cif',
   'pfile': 'pdb_ensemble/predictions/7gqt/7gqt/7gqt_out/7gqt_out.cif',
   'pocket': '5',
   'pocket_cog': [11.266744613647461, -2.0004360675811768, -5.426126956939697],
   'guide_cog': [6.413934230804443, -3.6677331924438477, -8.020732879638672]},
  '8yle': {'ciff': 'pdb_ens

In [46]:
inp_str = lambda strucmae, ligprep: f"""
#  Multiple input structures can be specified by adding additional
#  INPUT_FILE lines or including multiple structures in a single
#  file.
#
#  If beginning with an existing Pose Viewer file, simply specify
#  it as the INPUT_FILE (making sure the name ends in "_pv.mae"
#  or "_pv.maegz") and ensure that the first GLIDE_DOCKING stage
#  is commented out.  The ligand used in producing the Pose Viewer
#  file must also be provided to the second GLIDE_DOCKING stage,
#  using the LIGAND_FILE keyword.

INPUT_FILE	{strucmae}

# Prime Loop Prediction
#  Perform a loop prediction on the specified loop, including
#  side chains within the given distance.  Only return
#  structures within the specified energy range from the
#  lowest energy prediction, up to the maximum number of
#  conformations given.
#
#  Note: This stage is disabled by default.  Uncomment the
#   lines below and edit the fields appropriately to enable it.
#STAGE PRIME_LOOP
#  START_RESIDUE A:11
#  END_RESIDUE A:16
#  RES_SPHERE 7.5
#  MAX_ENERGY_GAP 30.0
#  MAX_STRUCTURES 5
#  USE_MEMBRANE no

STAGE GLIDE_DOCKING2
  BINDING_SITE ligand Z:899
  INNERBOX 15.0
  OUTERBOX 30.0
  LIGAND_FILE  {ligprep}
  LIGANDS_TO_DOCK all
  GRIDGEN_RECEP_CCUT  0.25
  GRIDGEN_RECEP_VSCALE 0.50
  GRIDGEN_FORCEFIELD OPLS_2005
  DOCKING_PRECISION SP
  DOCKING_LIG_CCUT  0.15
  DOCKING_CV_CUTOFF  100.0
  DOCKING_LIG_VSCALE 0.50
  DOCKING_POSES_PER_LIG 20
  DOCKING_FORCEFIELD OPLS_2005
  DOCKING_RINGCONFCUT 2.5
  DOCKING_AMIDE_MODE penal

STAGE COMPILE_RESIDUE_LIST
  DISTANCE_CUTOFF	5.0

STAGE PRIME_REFINEMENT
  NUMBER_OF_PASSES	1
  USE_MEMBRANE no
  OPLS_VERSION OPLS_2005

STAGE SORT_AND_FILTER
  POSE_FILTER	r_psp_Prime_Energy
  POSE_KEEP	30.0

STAGE SORT_AND_FILTER
  POSE_FILTER	r_psp_Prime_Energy
  POSE_KEEP	20#

STAGE GLIDE_DOCKING2
  BINDING_SITE ligand Z:999
  INNERBOX 10.0
  OUTERBOX auto
  LIGAND_FILE  {ligprep}
  LIGANDS_TO_DOCK self
  GRIDGEN_RECEP_CCUT  0.25
  GRIDGEN_RECEP_VSCALE 1.00
  GRIDGEN_FORCEFIELD OPLS_2005
  DOCKING_PRECISION SP
  DOCKING_LIG_CCUT  0.15
  DOCKING_CV_CUTOFF  0.0
  DOCKING_LIG_VSCALE 0.80
  DOCKING_POSES_PER_LIG 1
  DOCKING_FORCEFIELD OPLS_2005
  DOCKING_RINGCONFCUT 2.5
  DOCKING_AMIDE_MODE penal

STAGE SCORING
  SCORE_NAME  r_psp_IFDScore
  TERM 1.0,r_i_glide_gscore,0
  TERM 0.05,r_psp_Prime_Energy,1
  REPORT_FILE report.csv
"""

In [47]:
from schrodinger.structure import StructureReader, StructureWriter

In [69]:
for holo, holod in dockings.items():        
    if not ( glidedir / "ligpreps" / f"{holo}.mae" ).exists():
        outmae = prepare_ligands((glidedir.parent / "glide" / holo / f"{holo}.smi").absolute(), holo, glidedir / "ligpreps")
    
    for guide in ("pocket", "guide"):
        path = glidedir / f"{holo}_{guide}"
        path.mkdir(exist_ok=True)

        for apo, apod in holod.items():
            apopath = path / apo
            apopath.mkdir(exist_ok=True)
            if (apopath / f"{apo}-out.maegz").exists():
                continue

            cog_coords = apod[f"{guide}_cog"]
            strucmae = apopath / f"{apo}_center.mae"
            if holo == "8uk6" and apo != "8uk6":
                st = next(StructureReader((glidedir.parent / "glide" / holo / "preps" / f"{apo}_updated_prep.mae")))
            else:
                st = next(StructureReader((glidedir.parent / "glide" / holo / "preps" / f"{apo}_prep.mae")))
            a = st.addAtom("Du", *cog_coords)
            a.pdbres = "DUM"
            a.resnum = 899
            a.chain = "Z"
            StructureWriter(strucmae).append(st)
            StructureWriter(str(strucmae).replace(".mae", ".pdb")).append(st)
    
            ifd_inp = apopath / f"{apo}.inp"
            ifd_inp.write_text(
                inp_str(strucmae.name, outmae.absolute())
            )
        
            run(
                " ".join((
                    SCHRO + "/ifd",
                    ifd_inp.name,
                    "-NGLIDECPU 16 -NPRIMECPU 16 -NOLOCAL -HOST localhost -SUBHOST localhost -TMPLAUNCHDIR -WAIT"
                )),
                cwd=apopath, 
                shell=True
            )
            
            for rank, st in enumerate(
                sorted(
                    StructureReader(apopath / f"{apo}-out.maegz"), 
                    key=lambda st: st.property.get("r_psp_IFDScore")
                )[:5],
                1
            ):
                fname = apopath / f"{apo}_rank{rank}.pdb"
                with sstruct.StructureWriter(str(fname)) as w: w.append(st)

# OLD

# Functions

In [1]:
#!/usr/bin/env python
# blind_glide.py
"""
Blind docking with Glide using SiteMap site centers.

Pipeline:
  1) Protein preparation (prepwizard)
  2) Site detection (sitemap) -> site centers
  3) Build a Glide grid per site center
  4) Ligand preparation (ligprep + epik states)
  5) Dock ligands into each grid (glide)
  6) Collect top poses per grid and overall

Usage example (run with Schrodinger's Python):
  $SCHRODINGER/run python blind_glide.py \
      --receptor 1mj5.pdb \
      --ligand lig.sdf \
      --outdir run1 \
      --precision HTVS \
      --num_sites 5 \
      --inner 14 --outer 30

Notes:
- Requires a working Schrödinger 2025-2 installation (prepwizard, sitemap, ligprep, glide).
- "Blind" here means we auto-detect pockets and dock to each without prior knowledge.
"""

'\nBlind docking with Glide using SiteMap site centers.\n\nPipeline:\n  1) Protein preparation (prepwizard)\n  2) Site detection (sitemap) -> site centers\n  3) Build a Glide grid per site center\n  4) Ligand preparation (ligprep + epik states)\n  5) Dock ligands into each grid (glide)\n  6) Collect top poses per grid and overall\n\nUsage example (run with Schrodinger\'s Python):\n  $SCHRODINGER/run python blind_glide.py       --receptor 1mj5.pdb       --ligand lig.sdf       --outdir run1       --precision HTVS       --num_sites 5       --inner 14 --outer 30\n\nNotes:\n- Requires a working Schrödinger 2025-2 installation (prepwizard, sitemap, ligprep, glide).\n- "Blind" here means we auto-detect pockets and dock to each without prior knowledge.\n'

In [11]:
import argparse
import gzip
import os
import shutil
import subprocess as sp
import sys
import tempfile
from pathlib import Path
from typing import List, Tuple, Optional
import pandas as pd

In [3]:
# We'll read MAE/MAEGZ only if we need to compute centers from site points
try:
    from schrodinger import structure as sstruct
except Exception:
    sstruct = None

SCHRO = os.environ.get("SCHRODINGER") or "/opt/schrodinger2025-2"

In [4]:
sstruct

<module 'schrodinger.structure' from '/opt/schrodinger2025-2/internal/lib/python3.11/site-packages/schrodinger/structure/__init__.py'>

## General

In [15]:
def run(cmd: List[str], cwd: Optional[str] = None, **kwargs) -> None:
    print(f"[CMD] {cmd if 'shell' in kwargs else ' '.join(cmd)}")
    p = sp.run(cmd, check=True, cwd=cwd, text=True, capture_output=True, **kwargs)
    print(p.stdout)
    return


def which(tool: str) -> str:
    """Return full path to a Schrödinger CLI tool."""
    p = Path(SCHRO) / tool
    if p.exists():
        return str(p)
    # Some tools live at the root (e.g., $SCHRODINGER/prepwizard), others under subdirs.
    for sub in ["", "utilities", "shape_screen", "glide", "sitemap", "ligprep", "epik", "vsw"]:
        pp = Path(SCHRO) / sub / tool
        if pp.exists():
            return str(pp)
    return tool  # fallback on PATH


def ensure_outdir(d: Path):
    d.mkdir(parents=True, exist_ok=True)
    print(f"[INFO] Output directory: {d}")

## ProtPrep

In [16]:
cmd = [
    which("prepwizard"),
    "--help"
]
run(cmd)

[CMD] /opt/schrodinger2025-2/utilities/prepwizard --help
usage: $SCHRODINGER/utilities/prepwizard [options] inputfile outputfile
    Input file should be in Maestro or PDB format.
    Output file should be in Maestro or PDB format.
    

Required:
  in_file               Input structure file (Maestro or PDB)
  out_file              Output structure file (Maestro or PDB)

Preprocess:
  -nopreprocess         Skip pre-processing (useful if only ProtAssign or
                        Impref is desired)
  -reference_st_file REFERENCE_ST_FILE
                        File containing reference structure to align to.
  -reference_pdbid REFERENCE_PDBID
                        PDB ID of the structure to align to.
  -nobondorders         Don't assign bond orders to het groups
  -noccd                Don't use the Chemical Component Dictionary when
                        assigning bond orders.
  -assign_all_residues  Assign bond orders to all residues (by default
                        residues ar

In [27]:
def prepare_protein(infile: Path, out, cwd: Path) -> Path:
    """Protein Preparation Wizard (prepwizard) → MAE"""
    prep_out =  f"{out}_prep.mae" #outdir /
    if (cwd / prep_out).exists():
        return
    prepwizard = which("prepwizard")
    # Typical settings: assign bond orders, add H, optimize H-bond network, restrained minimization.
    cmd = [
        prepwizard,
        "-WAIT",
        "-assign_all_residues", #  Assign bond orders to all residues (by default residues are only processed if they only have single order bonds). If CCD is used, existing bond orders and formal charges will be overriden for residues that are present in the CCD.
        "-rehtreat", # Delete and re-add hydrogens (will reset PDB atom names)
        "-disulfides", # Create bonds to proximal Sulfurs (delete hydrogens as needed)
        "-mse", # Convert Selenomethionine residues to Methionines
        "-max_states 1", # Maximum number of het states to generate for each protein complex.
        "-captermini", # Add terminal oxygens to polypeptides, Cap termini
        # "-minimize_adj_h", # Energy minimize all adjustable hydrogens (titratable hydrogens, water hydrogens, hydroxyls, thiols, and ASN/GLN carboxamide hydrogens).
        "-f S-OPLS",
        "-epik_pH 7.4 -epik_pHt 2.0 -samplewater -include_epik_states -propka_pH 7.4  -rmsd 0.3 -watdist 5.0",
        
        # "-noprotassign",   # speed-up; set as needed
        # "-noimpref",       # we’ll still do a restrained minimization
        
        str(infile.absolute()),
        str(prep_out),
    ]
    run(" ".join(cmd), cwd=cwd, shell=True)
    return #prep_out

## SiteMap

In [19]:
cmd = [
    which("sitemap"),
    "--help"
]
run(cmd)

[CMD] /opt/schrodinger2025-2/sitemap --help
The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.
usage: 
        $SCHRODINGER/sitemap [options] -j <jobname> -prot <file.mae>
    or
        $SCHRODINGER/sitemap [options] <input_file.in>

$SCHRODINGER/sitemap carries out a site-finding job, then calculates SiteMaps
for each qualifying site-point grouping, and finally evaluates the SiteMap
results and summarizes the characteristics of the sites in a series of Maestro
properties. SiteMap can be invoked from Maestro or can be run from the command
line.

positional arguments:
  input_file            A file with lines such as "KEYWORD value", which are
                        equivalent to "-keyword value" command-line arguments.
                        (Except for job control options such as -HOST)

options:
  -j <jobname>, -job <jobname>, -jobname <jobname>
                        A unique d

In [ ]:
def run_sitemap(receptor_mae: Path, outdir: Path, num_sites: int) -> Path:
    """
    Run SiteMap and return the base name used by sitemap outputs.
    We’ll use --maxsites to limit.
    """
    sitemap = which("sitemap")
    base = outdir / "sitemap_job"
    cmd = [
        sitemap,
        "-WAIT",
        "-jobname", str(base.name),
        "-protein", str(receptor_mae),
        # "-sitebox 6", # default
        # "-compact_mode_threshold 800" # default
        # -maxsites 5 default
        "-verbosity 3",
        "-keepvolpts yes",
        # "-cleanup no -compress no -keepvolpts yes -writepot yes -keepvdw yes -keepelec yes -writestructs yes -writegrids yes",
        
        # "-site", "1",              # start site index; CLI will enumerate sites
        # "-maxsites", str(num_sites),
        # "-sitebox", "20.0",        # search box (~protein-centric); adjust if needed
        # "-writecsv",               # request CSV summary (contains per-site properties; often includes centers)
        # "-writePoints",            # write site points maegz (fallback to compute centers)
    ]
    run(" ".join(cmd), cwd=str(outdir), shell=True)

    # run(f"{SCHRO}/utilities/proplister {str(base.name)}_out.maegz -a -c -o {base.name}_site_properties.csv", shell=True)
    return base

### Centers

In [30]:
def centroid_of_points(points: List[Tuple[float,float,float]]) -> Tuple[float,float,float]:
    if not points:
        raise ValueError("No points to centroid.")
    n = len(points)
    sx = sum(p[0] for p in points)
    sy = sum(p[1] for p in points)
    sz = sum(p[2] for p in points)
    return (sx/n, sy/n, sz/n)


def parse_centers_from_volpts(job_dir: Path, jobname: str) -> List[Tuple[float, float, float]]:
    """
    Compute a centroid per site from the *_volpts.pdb files produced by -keepvolpts yes.
    Accepts common filename patterns:
      {jobname}_site{N}_volpts.pdb
      {jobname}_site{N}.volpts.pdb
      {jobname}_site{N}-volpts.pdb
      or any '*site*volpts.pdb' within the job directory.
    """
    import glob

    patterns = [
        f"{jobname}_site*_volpts.pdb",
        f"{jobname}_site*.volpts.pdb",
        f"{jobname}_site*-volpts.pdb",
        "*site*volpts.pdb",
    ]
    pdb_paths = []
    for pat in patterns:
        pdb_paths.extend([Path(p) for p in glob.glob(str(job_dir / pat))])

    centers: List[Tuple[float,float,float]] = []

    for pdb in sorted(set(pdb_paths)):
        pts: List[Tuple[float,float,float]] = []
        with pdb.open() as f:
            for line in f:
                if line.startswith(("ATOM","HETATM")):
                    # PDB columns: x(31–38), y(39–46), z(47–54) (1-based)
                    try:
                        x = float(line[30:38].strip())
                        y = float(line[38:46].strip())
                        z = float(line[46:54].strip())
                        pts.append((x,y,z))
                    except ValueError:
                        continue
        if pts:
            centers.append(centroid_of_points(pts))
    return centers


def collect_site_centers(sitemap_base: Path) -> List[Tuple[float,float,float]]: # (job_folder: Path, jobname: str)
    """
    Preferred order:
      1) proplister CSV from myjob_out.mae → parse_centers_from_props_csv
      2) fall back to centroids from *_volpts.pdb files
    """
    # csv_path = job_folder / f"{jobname}_props.csv"     # your own exported filename
    # # If you instead use the raw proplister output name:
    # if not csv_path.exists():
    #     alt = job_folder / f"{jobname}_props.tsv"
    #     if alt.exists():
    #         csv_path = alt
    #     else:
    #         # very common name when calling: proplister myjob_out.mae -a -c -o myjob_props.csv
    #         common = job_folder / f"{jobname}_props.csv"
    #         if common.exists():
    #             csv_path = common

    # centers = parse_centers_from_props_csv(csv_path)
    # if centers:
    #     return centers

    # Fallback: compute from volpts
    return parse_centers_from_volpts(sitemap_base, sitemap_base)

In [32]:
parse_centers_from_volpts(Path("Glide.prj/jobs/sitemap_test"), jobname="test")

[(93.78350855745711, 11.69895354523225, 26.4618875305624),
 (109.88894463667802, 16.39640830449828, 13.259522491349498),
 (83.3760526315787, 16.661157894736874, 29.032210526315755),
 (98.86999999999992, 9.624666666666652, 22.507583333333347),
 (86.04350746268638, 27.978895522388118, 40.05979104477609)]

## Grid

In [ ]:
def write_glide_grid_in(
    grid_in: Path,
    receptor_mae: Path,
    center: Tuple[float, float, float],
    inner: 10,#float,
    outer: 30#float
) -> None:
    x, y, z = center
    contents = f"""FORCEFIELD   OPLS_2005
GRID_CENTER   {x:.8f}, {y:.8f}, {z:.8f}
GRIDFILE   {grid_in.with_suffix('.zip')}
INNERBOX   {inner:.2f}, {inner:.2f}, {inner:.2f}
OUTERBOX   {outer:.2f}, {outer:.2f}, {outer:.2f}
RECEP_FILE   {receptor_mae}
"""
    grid_in.write_text(contents)

def build_grid(grid_in: Path) -> Path:
    glide = which("glide")
    run([glide, "-WAIT", str(grid_in)])
    grid_zip = grid_in.with_suffix(".zip")
    if not grid_zip.exists():
        raise RuntimeError(f"Grid not created: {grid_zip}")
    return grid_zip

## LigPrep

In [34]:
cmd = [
    which("ligprep"),
    "-help"
]
run(cmd)

[CMD] /opt/schrodinger2025-2/ligprep -help
usage: ligprep [options] (-ismi|-icsv|-imae|-isd) infile (-osd|-omae|-osmi|-ocsv) outfile

General:
  -h, -help             Print brief help message.
  -long_help            Print exhaustive help message.
  -inp <filename>       Read arguments from the supplied input file.
  -sif_docs             List supported simplified input file format (SIF)
                        keywords.

Ionization:
  -epik                 Use Epik Classic for ionization and tautomerization
                        (Recommended, overrides -i).
  -epikx                Use Epik for ionization and tautomerization
                        (Recommended, overrides -i).
  -emb, -epik_metal_binding
                        Run Epik with the metal_binding option so that states
                        appropriate for interactions with metal ions in
                        protein binding pockets are also generated.
  -i {0,1,2}            Ionization treatment: 0 - do not neutraliz

In [ ]:
def prepare_ligands(ligand_in: Path, outdir: Path) -> Path: #  pH: float = 7.4, dpH: float = 1.0
    """
    LigPrep with Epik states; outputs a single MAEGZ of prepared ligands.
    Accepts SDF/MAE/SMILES (SMILES must be in a .smi or via -ismi).
    """
    ligprep = which("ligprep")
    out_mae = outdir / "ligprep_out.maegz"

    ligprep_inp = outdir / "ligprep.inp"

    ligprep_inp.write_text(f"""
INPUT_FILE_NAME   {str(ligand_in)}
MAX_ATOMS   500
FORCE_FIELD   16
EPIK   yes
EPIKX   no
EPIK_METAL_BINDING   no
INCLUDE_ORIGINAL_STATE   no
DETERMINE_CHIRALITIES   no
IGNORE_CHIRALITIES   yes
NUM_STEREOISOMERS   32
OUT_MAE   {out_mae}
    """)

    # Detect SMILES input by extension
    # is_smiles = ligand_in.suffix.lower() in [".smi", ".smiles", ".can"] # has to be .smi
    cmd = [ligprep, "-WAIT", "-inp", str(ligprep_inp)]
    # Epik-like states around target pH
    # cmd += ["-ph", f"{pH}", "-pht", f"{dpH}", "-s", "1"]  # 1 conformer per state (adjust as needed)
    run(cmd)
    return out_mae

## TEST

In [9]:
bioemuensdir = Path("bioemu_8uk6/msmclust/dihedrals")

In [12]:
bioemuensresults = pd.read_pickle(bioemuensdir / "predictions.pkl")
bioemuensresults

{'frame_0': {'pocket': 'pocket1',
  'overlap': 0.9047619047619048,
  'merge':    label_comp_id label_asym_id pdbx_PDB_ins_code auth_asym_id  \
  0            GLY             A                 ?            A   
  1            PHE             A                 ?            A   
  2            LEU             A                 ?            A   
  3            HIS             A                 ?            A   
  4            HIS             A                 ?            A   
  5            GLY             A                 ?            A   
  6            THR             A                 ?            A   
  7            ILE             A                 ?            A   
  8            MET             A                 ?            A   
  9            LYS             A                 ?            A   
  10           ILE             A                 ?            A   
  11           TRP             A                 ?            A   
  12           ARG             A                 ?   

In [29]:
outdir = Path("glide/8uk6_prep")
outdir.mkdir(exist_ok=True)

for fn, f in [("8uk6", "dldocking/diffdock-pocket/8uk6/8uk6.pdb")] + [(frame, f"dldocking/diffdock-pocket/8uk6/{frame}.pdb") for frame in bioemuensresults]:
    prepare_protein(Path(f), out=fn, cwd=outdir)

In [33]:
seqsf = f"cofolding/seqs.pkl"

seqs = pd.read_pickle(seqsf)

In [43]:
outdir = Path("glide")

smi = outdir / "xglide.smi"
smi.write_text(seqs["8uk6"]["smiles"])

xglide_inp = outdir / "xglide.inp"

xglide_inp.write_text(f"""
RECEPTOR	{'8uk6_prep'}
ALIGN	FALSE
LIGAND	{smi.name}
GRIDGEN_GRID_CENTER	SELF
SITEMAP	TRUE
SITEMAP_MAXSITES    5
GRIDGEN_INNERBOX	10
GRIDGEN_OUTERBOX	26.0
LIGPREP	TRUE
LIGPREP_EPIK	TRUE
DOCK_PRECISION	SP
DOCK_WRITE_XP_DESC	FALSE
DOCK_POSE_OUTTYPE	poseviewer
NATIVEONLY	FALSE
DOCK_LIG_VSCALE	0.80
SKIP_DOCKING	FALSE
GOOD_RMSD	2.0
GENERATE_TOP_COMPLEXES	5
    """)

355

In [45]:
run(
    " ".join((
        SCHRO + "/run xglide.py",
        xglide_inp.name,
        "-WAIT -HOST localhost:1 -NJOBS 16 -TMPLAUNCHDIR"
    )),
    cwd=Path("glide"), 
    shell=True
)

[CMD] /opt/schrodinger2025-2/run xglide.py xglide.inp -WAIT -HOST localhost:1 -NJOBS 16 -TMPLAUNCHDIR
JobId: avogadro-0-68d6cd73



In [50]:
import re

In [55]:
# change to your .maegz
outdir = Path("glide/pdbs"); outdir.mkdir(exist_ok=True)
with sstruct.StructureReader("glide/xglide_topcomplexes.maegz") as r:
    for i, st in enumerate(r, 1):
        title = re.sub(r"[^\w\-]+","_", st.title)[:30] if st.title else f"pose{i}"
        # score = st.property.get("r_i_docking_score") or st.property.get("r_glide_gscore")
        grid = st.property.get("s_i_glide_gridfile", "") # 'xglide__frame_3_prep_site1__grid'
        frame, site = re.search(r"xglide__([^_]+(?:_\d+)?)_prep_(site\d+)", grid).groups()
        fname = f"{'8uk6'}_{frame}_pose{i}_{site}"
        # if score is not None: fname += f"_score_{float(score):+.2f}"
        with sstruct.StructureWriter(str(outdir/f"{fname}.pdb")) as w: w.append(st)

'xglide__8uk6_prep_site5__grid'

In [46]:
test = sstruct.MaestroReader("glide/xglide_topcomplexes.maegz")
test

In [48]:
test = test.read()
test

Structure(2)

In [ ]:
test.

In [ ]:
test.

In [31]:
SCHRO / "a"

TypeError: unsupported operand type(s) for /: 'str' and 'str'

## Glide

In [ ]:
def write_glide_dock_in(
    dock_in: Path,
    grid_zip: Path,
    lig_mae: Path,
    precision: str = "HTVS",
    nposes: int = 5
) -> None:
    """
    precision: HTVS | SP | XP
    """
    precision = precision.upper()
    if precision not in {"HTVS", "SP", "XP"}:
        raise ValueError("precision must be one of: HTVS, SP, XP")

    contents = f"""DOCKING
GRIDFILE {grid_zip}
LIGANDFILE {lig_mae}
PRECISION {precision}
POSES_PER_LIG {nposes}
WRITE_XP_DESC 1
NREPORT 25
"""
    dock_in.write_text(contents)


def dock(grid_zip: Path, lig_mae: Path, outdir: Path, precision: str) -> Path:
    glide = which("glide")
    jobname = f"dock_{grid_zip.stem}"
    dock_in = outdir / f"{jobname}.in"
    write_glide_dock_in(dock_in, grid_zip, lig_mae, precision=precision, nposes=5)
    run([glide, "-WAIT", str(dock_in)])
    out_mae = outdir / f"{jobname}_pv.maegz"
    if not out_mae.exists():
        # Some versions write <jobname>_lib.maegz or _raw.maegz; collect any *_pv.maegz fallback
        candidates = list(outdir.glob(f"{jobname}*_pv.mae*"))
        if candidates:
            return candidates[0]
        raise RuntimeError(f"Docking output not found for {grid_zip}")
    return out_mae


def pick_top_pose_from_mae(mae_file: Path, out_pose: Path) -> bool:
    """
    Copy best-scoring pose (first structure) into out_pose (MAE).
    """
    if sstruct is None:
        print("[WARN] schrodinger.structure not available; skipping top-pose extraction.")
        return False
    sts = list(sstruct.StructureReader(str(mae_file)))
    if not sts:
        return False
    with sstruct.StructureWriter(str(out_pose)) as w:
        w.append(sts[0])
    return True


def merge_best_poses(pose_files: List[Path], merged_out: Path) -> None:
    if sstruct is None:
        # fallback: concatenate binary files is not safe; skip
        print("[WARN] schrodinger.structure not available; skipping merged poses.")
        return
    with sstruct.StructureWriter(str(merged_out)) as w:
        for pf in pose_files:
            try:
                sts = list(sstruct.StructureReader(str(pf)))
                if sts:
                    w.append(sts[0])
            except Exception:
                pass


def main():
    ap = argparse.ArgumentParser(description="Blind docking with Glide via SiteMap pockets.")
    ap.add_argument("--receptor", required=True, help="Protein structure (PDB/MAE).")
    ap.add_argument("--ligand", required=True, help="Ligand file (SDF/MAE/SMI).")
    ap.add_argument("--outdir", required=True, help="Output directory.")
    ap.add_argument("--num_sites", type=int, default=5, help="Number of SiteMap sites to use.")
    ap.add_argument("--inner", type=float, default=14.0, help="Glide inner box (Å).")
    ap.add_argument("--outer", type=float, default=30.0, help="Glide outer box (Å).")
    ap.add_argument("--precision", default="HTVS", choices=["HTVS", "SP", "XP"], help="Docking precision.")
    ap.add_argument("--ph", type=float, default=7.4, help="LigPrep target pH.")
    ap.add_argument("--dph", type=float, default=1.0, help="LigPrep pH tolerance (±).")
    args = ap.parse_args()

    outdir = Path(args.outdir).absolute()
    ensure_outdir(outdir)

    receptor_in = Path(args.receptor).absolute()
    ligand_in = Path(args.ligand).absolute()

    # 1) Prepare protein
    receptor_prep = prepare_protein(receptor_in, outdir)

    # 2) SiteMap
    sm_base = run_sitemap(receptor_prep, outdir, args.num_sites)
    centers = collect_site_centers(sm_base)
    if not centers:
        print("[ERROR] No site centers found by SiteMap. Consider increasing --num_sites or adjusting --sitebox.", file=sys.stderr)
        sys.exit(1)

    centers = centers[: args.num_sites]
    print(f"[INFO] Using {len(centers)} site centers for grid generation.")

    # 3) Build grids
    grid_zips: List[Path] = []
    grids_dir = outdir / "grids"
    grids_dir.mkdir(exist_ok=True)
    for i, c in enumerate(centers, 1):
        grid_in = grids_dir / f"grid_site{i}.in"
        write_glide_grid_in(grid_in, receptor_prep, c, args.inner, args.outer)
        gz = build_grid(grid_in)
        grid_zips.append(gz)

    # 4) Ligand prep
    lig_mae = prepare_ligands(ligand_in, outdir, pH=args.ph, dpH=args.dph)

    # 5) Dock per grid
    poses_dir = outdir / "poses"
    poses_dir.mkdir(exist_ok=True)
    best_pose_files: List[Path] = []
    for gz in grid_zips:
        dock_out = dock(gz, lig_mae, poses_dir, args.precision)
        top_pose = poses_dir / f"best_{gz.stem}.mae"
        ok = pick_top_pose_from_mae(dock_out, top_pose)
        if ok:
            best_pose_files.append(top_pose)
        else:
            # If picking failed, still keep the full pv file in the merged set
            best_pose_files.append(dock_out)

    # 6) Merge best poses
    merged = outdir / "best_poses_all_sites.mae"
    merge_best_poses(best_pose_files, merged)

    print("\n[DONE]")
    print(f"Prepared receptor: {receptor_prep}")
    print(f"Site centers used: {len(centers)}")
    print(f"Grids: {len(grid_zips)} → {grids_dir}")
    print(f"Ligands (LigPrep): {lig_mae}")
    print(f"Docking outputs: {poses_dir}")
    if merged.exists():
        print(f"Merged best poses: {merged}")
    else:
        print("Merged best poses skipped (missing schrodinger.structure).")


if __name__ == "__main__":
    main()